# Immo Eliza – House Price Prediction (Baseline)

This notebook builds a **first baseline regression model** to predict Belgian real estate prices
for Immo Eliza.

Goals of this notebook:
- Load and explore the dataset.
- Build a **very first baseline** using only numerical features.
- Train and evaluate a **Linear Regression** model.
- Set up a clear structure we can later upgrade into a professional ML pipeline.


In [70]:
# Standard libraries
import os

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [71]:
# Load raw data and apply basic cleaning (v1)

DATA_PATH = "../data/raw/immo_eliza_raw.csv" 
TARGET_COL = "Price"

df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)

# Drop duplicate rows
df = df.drop_duplicates()

# Clean target: convert Price to numeric
# 1. Make it a string
price_str = df[TARGET_COL].astype(str)

# 2. Remove everything that is NOT a digit
price_str = price_str.str.replace(r"[^\d,\.]", "", regex=True)

# 3. remove dots, replace comma by dot
price_str = price_str.str.replace(".", "", regex=False)   # remove thousands separator
price_str = price_str.str.replace(",", ".", regex=False)  # convert decimal comma to dot

# 4. Empty strings -> NaN
price_str = price_str.replace("", np.nan)

# 5. Convert to float
df[TARGET_COL] = price_str.astype(float)

# 6. Drop rows where target is missing after cleaning
df = df.dropna(subset=[TARGET_COL])

print("After cleaning Price and dropping missing target:", df.shape)
display(df[[TARGET_COL]].head())


Original shape: (16309, 26)
After cleaning Price and dropping missing target: (15725, 26)


,Price
0,175000.0
1,415000.0
2,399000.0
3,229000.0
4,320000.0


In [72]:
# Separate target and features

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataframe.")

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
print("Number of numeric features:", len(numeric_cols))
print("Number of categorical features:", len(categorical_cols))


Numeric features: ['Number of bedrooms', 'Number of garages', 'Number of bathrooms', 'Number of showers', 'Number of toilets', 'Number of facades']
Categorical features: ['url', 'Property ID', 'State of the property', 'Availability', 'Livable surface', 'Furnished', 'Attic', 'Garage', 'Kitchen equipment', 'Kitchen type', 'Type of heating', 'Type of glazing', 'Elevator', 'Garden', 'Surface garden', 'Terrace', 'Surface terrace', 'Total land surface', 'Swimming pool']
Number of numeric features: 6
Number of categorical features: 19


In [73]:
# Baseline: use only numeric features and drop remaining NaNs in features

X_num = X[numeric_cols].copy()

print("Shape before dropping NaNs (numeric-only):", X_num.shape)

baseline_df = pd.concat([X_num, y], axis=1)
baseline_df = baseline_df.dropna()  # drop rows with NaNs in features or target

print("Shape after dropping NaNs (baseline dataset):", baseline_df.shape)

X_num_clean = baseline_df.drop(columns=[TARGET_COL])
y_clean = baseline_df[TARGET_COL]


Shape before dropping NaNs (numeric-only): (15725, 6)
Shape after dropping NaNs (baseline dataset): (968, 7)


In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X_num_clean,
    y_clean,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)


Train shape: (774, 6) (774,)
Test shape: (194, 6) (194,)


In [75]:
# Create and train the baseline Linear Regression model
linreg = LinearRegression()

linreg.fit(X_train, y_train)

print("Number of features used:", X_train.shape[1])
print("Intercept:", linreg.intercept_)


Number of features used: 6
Intercept: 2228.317990775453


In [76]:
# Predict
y_pred_train = linreg.predict(X_train)
y_pred_test = linreg.predict(X_test)

def regression_metrics(y_true, y_pred, prefix=""):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{prefix}MAE : {mae:,.2f}")
    print(f"{prefix}MSE : {mse:,.2f}")
    print(f"{prefix}RMSE: {rmse:,.2f}")
    print(f"{prefix}R²  : {r2:,.4f}")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

print("=== Train performance ===")
train_metrics = regression_metrics(y_train, y_pred_train, prefix="[Train] ")

print("\n=== Test performance ===")
test_metrics = regression_metrics(y_test, y_pred_test, prefix="[Test] ")


=== Train performance ===
[Train] MAE : 215,577.73
[Train] MSE : 123,338,323,134.71
[Train] RMSE: 351,195.56
[Train] R²  : 0.3813

=== Test performance ===
[Test] MAE : 368,775.20
[Test] MSE : 5,079,715,916,845.21
[Test] RMSE: 2,253,822.51
[Test] R²  : -24.3137


## Is the model overfitting?

We compare train and test performance:

- If **train error is much lower** than test error and
  train R² is much higher than test R², the model is likely **overfitting**.
- If performance is similar on both, the model is probably **generalizing** decently.


In [77]:
print("Train R²:", train_metrics["R2"])
print("Test  R²:", test_metrics["R2"])

Train R²: 0.38133307937572114
Test  R²: -24.313714533316812


In [78]:
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coef": linreg.coef_
})
coefs["abs_coef"] = coefs["coef"].abs()
coefs.sort_values("abs_coef", ascending=False).head(20)


,feature,coef,abs_coef
2,Number of bathrooms,193789.048026,193789.048026
4,Number of toilets,100729.675827,100729.675827
5,Number of facades,78766.466622,78766.466622
3,Number of showers,-65155.841393,65155.841393
0,Number of bedrooms,-46042.168147,46042.168147
1,Number of garages,22808.281053,22808.281053


## Next steps (towards a professional pipeline)

This initial baseline attempt revealed that the model does not generalize yet:

Current results from the baseline Linear Regression:
- **Train MAE: ~215,577.73**
- **Test MAE: ~368,775.20**
- **Train R²: 0.3813**
- **Test R²: -24.3137**

While the train score shows that the model captures some signal, the extremely low test score indicates a major need for:
- **Proper feature preprocessing**
- **Stable reusable pipelines**
- **Hidden numeric fields stored as text still needing conversion**
- **Outlier impact reduction**
- **Non-linear model comparison**

To move towards a reusable solution, I will:

1. Build a **reusable preprocessing pipeline** that applies:
   - NaN imputation (numerical + categorical)
   - One-Hot Encoding for categorical features
   - Standard Scaling for numerical comfort features
2. Wrap preprocessing + model into a single **scikit-learn Pipeline** for reuse in scripts and saved models.
3. Train and compare **three stable regression models**:
   - **Linear Regression** (baseline signal estimator)
   - **Random Forest Regressor** (non-linear model capturing feature interactions)
   - **Support Vector Regression (SVR)** (stable non-linear fallback model)
4. Evaluate **overfitting vs generalization** by comparing train and test errors and R² scores.
5. Export the final pipeline + preprocessing system using **joblib** so it can be reused by:
   - `train.py`
   - `predict.py` with dummy new house data
6. Update the repository accordingly and document the full approach + results in the README.



In [93]:
# Convert surface-related columns from text to numeric 

surface_cols = [
    "Livable surface",
    "Surface garden",
    "Surface terrace",
    "Total land surface",
]

for col in surface_cols:
    if col in df.columns:
        # 1. Convert to string
        s = df[col].astype(str)

        # 2. Remove everything that is NOT digit, comma or dot
        #    (removes m², spaces, strange chars)
        s = s.str.replace(r"[^\d,\.]", "", regex=True)

        # 3. Treat dot as thousands separator and comma as decimal
        s = s.str.replace(".", "", regex=False)
        s = s.str.replace(",", ".", regex=False)

        # 4. Empty strings -> NaN
        s = s.replace("", np.nan)

        # 5. Convert to float
        df[col] = pd.to_numeric(s, errors="coerce")

print("Converted surface columns to numeric ")
df[surface_cols].head()

Converted surface columns to numeric 


,Livable surface,Surface garden,Surface terrace,Total land surface
0,5100.0,NaN,NaN,NaN
1,7000.0,NaN,2000.0,NaN
2,12900.0,NaN,NaN,NaN
3,8200.0,NaN,800.0,NaN
4,10600.0,NaN,600.0,NaN


In [94]:
# Separate target and features (after all cleaning)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataframe.")

y = df[TARGET_COL]

# Drop ID-like columns from X
drop_cols = ["url", "Property ID"]
X = df.drop(columns=[TARGET_COL] + [c for c in drop_cols if c in df.columns])

# Recompute numeric and categorical columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
print("Number of numeric features:", len(numeric_cols))
print("Number of categorical features:", len(categorical_cols))

Numeric features: ['Number of bedrooms', 'Livable surface', 'Number of garages', 'Number of bathrooms', 'Number of showers', 'Number of toilets', 'Number of facades', 'Surface garden', 'Surface terrace', 'Total land surface']
Categorical features: ['State of the property', 'Availability', 'Furnished', 'Attic', 'Garage', 'Kitchen equipment', 'Kitchen type', 'Type of heating', 'Type of glazing', 'Elevator', 'Garden', 'Terrace', 'Swimming pool']
Number of numeric features: 10
Number of categorical features: 13


In [95]:
# Re-align y with X in case rows were dropped earlier
y_aligned = y.loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_aligned,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)



Train shape: (12580, 23) (12580,)
Test shape: (3145, 23) (3145,)


In [84]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR  
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Define models
model_dict = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42),
    "Support Vector Regression": SVR()  
}

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
results = []

for name, model in model_dict.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "MAE Train": mean_absolute_error(y_train, y_train_pred),
        "MAE Test": mean_absolute_error(y_test, y_test_pred),
        "RMSE Train": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "RMSE Test": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "R2 Train": r2_score(y_train, y_train_pred),
        "R2 Test": r2_score(y_test, y_test_pred),
        "Pipeline": pipe
    })

# Comparison table
compare = pd.DataFrame([
    {k: v for k, v in r.items() if k != "Pipeline"} for r in results
])

display(compare)
print("\nPipelines trained and ready to be compared.")


,Model,MAE Train,MAE Test,RMSE Train,RMSE Test,R2 Train,R2 Test
0,Linear Regression,91.677979,127068.686211,152.535583,285246.252654,1.000000,0.274934
1,Random Forest,38820.263579,103761.445668,103916.854365,274792.899932,0.899945,0.327103
2,Support Vector Regression,159826.084612,162596.030256,333430.120321,340455.980683,-0.030093,-0.032903



Pipelines trained and ready to be compared.


## Model evaluation (v2) and roadmap to iteration v3

The dataset was loaded, cleaned (duplicates removed, target converted to numeric, hidden text-numerical features converted to float), and split into training and testing sets:

- **Training set**: 12,580 samples, 23 features
- **Test set**: 3,145 samples, 23 features

### Performance summary (v2)
| Model | Train MAE | Test MAE | Train RMSE | Test RMSE | Train R² | Test R² |
|---|---:|---:|---:|---:|---:|---:|
| Linear Regression | 12.06 | 134,091.25 | 79.42 | 289,743.06 | 0.9999 | 0.2518 |
| Random Forest | 35,977.51 | 97,814.23 | 100,029.95 | 263,698.23 | 0.9073 | 0.3803 |
| Support Vector Regression | 159,818.41 | 162,589.24 | 333,415.33 | 340,455.98 | -0.032 | -0.032 |

### Interpretation
- The **Random Forest model currently performs best on test data (MAE ~97,814 | R² ~0.38)** and serves as our strongest model to iterate on.
- **Linear Regression shows unrealistically low error on train** and a **large drop on test**, indicating data leakage or instability from extreme scaling or outliers.
- **SVR is not extracting enough signal and shows clear underfitting** at this stage.

---

## Next iteration focus (v3)

To evolve this into a reusable, professional solution, I will:

1. **Feature refinement**
   - Remove non-predictive ID columns (e.g., `url`, `Property ID`).
   - Ensure all usable predictors are properly typed (numerical for surfaces, land, garden, terrace, etc.).
   - Investigate a **correlation matrix** to keep only features that contribute meaningful signal to Price.
   - Detect and handle **multicollinearity** and redundant features.

2. **Pipeline upgrade**
   Replace simple cleaning decisions by a reusable preprocessing system that contains:  
   ✔ numerical imputation (median or KNN-based imputer if needed)  
   ✔ categorical imputation (most frequent)  
   ✔ One-Hot Encoding for true categoricals  
   ✔ scaling (StandardScaler or RobustScaler based on outlier behavior)  
   ✔ optional log-transform on skewed numerical predictors or target for stability  

3. **Model improvement**
   Iterate by:
   - tuning **Random Forest complexity** (trees, depth, min sample leaf, feature sampling strategy)
   - evaluating whether **XGBoost can be reintroduced** as model 3 once libraries are verified stable
   - optionally testing Gradient Boosting regressors if needed

4. **Evaluation discipline**
   - Compute **MAE, MSE, RMSE, R² on both train and test** using the *same pipeline*.
   - Compare train vs. test: if train ≫ test → improve signal or allow complexity.
     If train ≪ test → reduce complexity, rescale or cap outliers.
   - Deliver an **explicit overfitting judgement** using metric gaps, not intuition.

5. **Reusable artifacts**
   - Save the preprocessing pipeline + best model using **joblib** into the `models/` directory
   - Implement:
     - `train.py` → training + saving
     - `predict.py` → loading + inference on *new dummy data*
     - dummy new house data example

6. **Documentation impact**
   - Write a **clean, non-negotiable README.md** explaining:
     - goal
     - approach
     - results
     - how to run the project
     - how to train
     - how to predict
   - Add reproducible execution steps (dependency install, notebook run, script usage)




# Version 3 – Advanced Pipeline and Tuned Random Forest


In [96]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np


In [97]:
def build_preprocessor(numeric_cols, categorical_cols):
    """Build a reusable preprocessing pipeline for numeric and categorical features."""
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

    return preprocessor


def get_base_models(random_state=42):
    """Return the base models we want to compare."""
    return {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1
        ),
        "Support Vector Regression": SVR()
    }


def evaluate_pipeline(pipe, X_train, X_test, y_train, y_test):
    """Train a pipeline, evaluate on train and test, and return metrics."""
    pipe.fit(X_train, y_train)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return {
        "MAE Train": mae_train,
        "MAE Test": mae_test,
        "RMSE Train": rmse_train,
        "RMSE Test": rmse_test,
        "R2 Train": r2_train,
        "R2 Test": r2_test
    }


In [98]:
# Build preprocessor from the final numeric/categorical lists
preprocessor_v3 = build_preprocessor(numeric_cols, categorical_cols)

# Train/test split (using aligned y as before)
y_aligned = y.loc[X.index]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_aligned,
    test_size=0.2,
    random_state=42
)

base_models = get_base_models(random_state=42)

v3_results = []

for name, model in base_models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor_v3),
        ("model", model)
    ])

    metrics = evaluate_pipeline(pipe, X_train, X_test, y_train, y_test)
    metrics["Model"] = name
    metrics["Pipeline"] = pipe
    v3_results.append(metrics)

# Turn into a clean DataFrame for comparison
v3_compare = pd.DataFrame([
    {k: v for k, v in res.items() if k != "Pipeline"} for res in v3_results
])

display(v3_compare.sort_values("MAE Test"))


,MAE Train,MAE Test,RMSE Train,RMSE Test,R2 Train,R2 Test,Model
1,41488.237832,97050.738217,105716.818546,223956.802604,0.896449,0.553043,Random Forest
0,133077.496241,134171.901686,290054.196046,289765.290849,0.220484,0.251778,Linear Regression
2,159796.995021,162575.626329,333389.635198,340419.827858,-0.029843,-0.032684,Support Vector Regression


In [99]:
# Focus on Random Forest for tuning
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.5]
}

rf_pipe = Pipeline([
    ("preprocess", preprocessor_v3),
    ("model", rf)
])

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_distributions,
    n_iter=15,  # keep this small for time
    scoring="neg_mean_absolute_error",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print("Best CV MAE:", -rf_search.best_score_)


Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best RF params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': 20}
Best CV MAE: 100785.74957070887


In [100]:
best_rf_pipe = rf_search.best_estimator_

best_rf_metrics = evaluate_pipeline(best_rf_pipe, X_train, X_test, y_train, y_test)
best_rf_metrics


{'MAE Train': 66782.55206206175,
 'MAE Test': 97186.25407812932,
 'RMSE Train': np.float64(168208.2649235376),
 'RMSE Test': np.float64(206820.90257674956),
 'R2 Train': 0.7378430073416913,
 'R2 Test': 0.6188231660487178}

In [101]:
import os

os.makedirs("../models", exist_ok=True)

MODEL_PATH = "../models/immo_eliza_random_forest.joblib"

joblib.dump(best_rf_pipe, MODEL_PATH)

print(f"Saved best Random Forest pipeline to: {MODEL_PATH}")


Saved best Random Forest pipeline to: ../models/immo_eliza_random_forest.joblib


## Final model – Random Forest (v3)

After converting hidden numerical features, building a reusable preprocessing
pipeline and running a light hyperparameter search on the Random Forest model,
the final selected model is:

> **Random Forest Regressor wrapped in a scikit-learn Pipeline  
> (preprocessing + model saved as a single object).**

### v3 base comparison (before tuning)

Using the common preprocessing pipeline:

- **Random Forest** clearly outperformed the other models:
  - Train MAE ≈ 41,488  
  - Test MAE ≈ 97,051  
  - Train R² ≈ 0.896  
  - Test R² ≈ 0.553

Linear Regression and SVR either underfit or showed much weaker generalisation,
so Random Forest was chosen for further tuning.

### Tuned Random Forest (final model)

After a small `RandomizedSearchCV` on key hyperparameters, the final model
achieves:

| Metric    | Train | Test |
|-----------|------:|-----:|
| MAE       | ~66,783 | ~97,186 |
| RMSE      | ~168,208 | ~206,821 |
| R²        | 0.738 | 0.619 |

**Interpretation**

- The model captures a substantial amount of the variance in prices  
  (**R² ≈ 0.62 on unseen data**), which is strong for noisy real-estate data.
- Train and test scores are reasonably close, indicating that the model
  generalises well and is **not heavily overfitting**.
- Errors are now in a realistic range given Belgian housing prices and the
  limited set of available features.

This tuned Random Forest pipeline is saved to:

```text
models/immo_eliza_random_forest.joblib
